# Assessment context playground

This notebook uses the assessment simulator, the same production schema compiler used by training and serving, and the selected comparison bundle. Its data is synthetic and is not evidence of institutional accuracy. Run the staged comparison in `compare_assessment_context.py` first; this notebook does not train a model.


In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys

# Work whether Jupyter starts at the repository root or inside examples/.
ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "services" / "ai" / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter somewhere inside the APMS repository")
sys.path.insert(0, str(ROOT / "services" / "ai" / "src"))

from apms_ai.assessment_synthetic import AssessmentSyntheticConfig, generate_assessment_examples
from apms_ai.assessment_context import ASSESSMENT_CONTEXT_FEATURE_NAMES, compile_assessment_contexts
from apms_ai.lifecycle import load_bundle
from apms_ai.inference import model_from_bundle, predict_example

ARTIFACTS = ROOT / "services/ai/artifacts/assessment-context-20261020/horizon-v2"
profile = json.loads((ROOT / "services/ai/examples/assessment-context-comparison.json").read_text())
example = generate_assessment_examples(AssessmentSyntheticConfig(groups=1, seed=20261020))[0]
print("learner group:", example.group)
print("assessment instances:", len(example.assessment_instances))
print("observed assessment results:", len(example.assessment_results))
print("forecast queries:", len(example.queries))

## Inspect the shared input contract

References join a target assessment to its schema entry and prior observed results. The compiler returns a fixed list of numeric features in query order; it does not copy learner or assessment references into model features.

In [ ]:
print('first assessment instances:')
for item in example.assessment_instances[:4]:
    print({key: item[key] for key in ('assessment_ref', 'assessment_type_ref', 'sequence', 'maximum_score', 'scoring', 'period_ref', 'group_ref', 'relative_day')})
print('first observed results:', example.assessment_results[:4])
print('first queries:', example.queries[:6])

context_rows = compile_assessment_contexts(example)
first_assessment_query = next(i for i, query in enumerate(example.queries) if 'assessment_ref' in query)
context = dict(zip(ASSESSMENT_CONTEXT_FEATURE_NAMES, context_rows[first_assessment_query]))
print('context for the first assessment query:')
print(context)

## Train and evaluate with the staged runner

The runner freezes one dataset, trains both context ablations on the same group split, selects using validation-only rules, and calibrates the selection using only the calibration role. Before final testing, complete the runtime, gateway, ONNX, and dynamic-parity checks and require a passing `validation-gates.json`. Use a fresh output directory for a new run; these commands do not overwrite the frozen comparison in `horizon-v2`. Run from the repository root:

```text
python services/ai/examples/compare_assessment_context.py prepare --output services/ai/artifacts/assessment-context-20261020/playground-new-run
python services/ai/examples/compare_assessment_context.py train --output services/ai/artifacts/assessment-context-20261020/playground-new-run --name context_control
python services/ai/examples/compare_assessment_context.py train --output services/ai/artifacts/assessment-context-20261020/playground-new-run --name assessment_context_v1
python services/ai/examples/compare_assessment_context.py select --output services/ai/artifacts/assessment-context-20261020/playground-new-run
python services/ai/examples/compare_assessment_context.py calibrate --output services/ai/artifacts/assessment-context-20261020/playground-new-run
python services/ai/examples/verify_assessment_context_runtime.py --experiment services/ai/artifacts/assessment-context-20261020/playground-new-run --attempt playground-runtime-review
python services/ai/examples/compare_assessment_context.py final --output services/ai/artifacts/assessment-context-20261020/playground-new-run
```

## Forecast with a selected bundle

This cell calls the same bundle loader and inference API used by the application. It strips labels from the copy passed to inference, then displays one raw numeric assessment forecast and one categorical forecast when available.

In [ ]:
selection_path = ARTIFACTS / "selection.json"
calibrated_path = ARTIFACTS / "selected-calibrated"
if calibrated_path.exists():
    bundle_path = calibrated_path
    bundle_status = "selected and calibrated"
elif selection_path.exists():
    selection = json.loads(selection_path.read_text())
    bundle_path = ARTIFACTS / selection["selected"]
    bundle_status = "selected; calibration has not been run"
else:
    bundle_path = None
    bundle_status = "no validation-selected bundle is available yet"

if bundle_path is None:
    print(bundle_status, "Run the staged commands above.")
else:
    state, metadata = load_bundle(bundle_path)
    model = model_from_bundle(state, metadata)
    result = predict_example(state, metadata, replace(example, labels=[]), model=model)
    assessment_predictions = [
        (query, prediction)
        for query, prediction in zip(example.queries, result["predictions"])
        if "assessment_ref" in query
    ]
    numeric = next(((query, prediction) for query, prediction in assessment_predictions
                    if query["field"] == "assessment_score"), None)
    categorical = next(((query, prediction) for query, prediction in assessment_predictions
                        if query["field"] == "categorical_score"), None)
    print("bundle status:", bundle_status)
    print("bundle:", bundle_path)
    print("assessment forecast count:", len(assessment_predictions))
    print("linear assessment_score forecast:", numeric)
    print("value-mapped categorical_score forecast:", categorical)